# 🚀 Google Colab GPU - Islamic Knowledge Harvester

**Purpose:** Process transcript chunks using FREE Tesla T4 GPU

**Tasks:**
- Summarize each transcript chunk (1-2 sentences)
- Classify into Islamic topics
- Generate confidence scores

---

## ⚠️ IMPORTANT: Before Running

1. **Enable GPU Runtime:**
   - Click `Runtime` → `Change runtime type`
   - Select `GPU` → `T4` (free tier)
   - Click `Save`

2. **Upload File:**
   - Upload `chunks.json` from your computer
   - File should be in `/content/` directory

3. **Processing Time:**
   - ~100 chunks: 2-3 minutes
   - ~500 chunks: 10-15 minutes
   - ~1000 chunks: 20-30 minutes

---

## Step 1: Install Dependencies

In [ ]:
!pip install transformers accelerate sentencepiece torch --quiet
print("✓ Dependencies installed")

## Step 2: Verify GPU & Upload Files

In [ ]:
import torch
print(f"✓ GPU Available: {torch.cuda.is_available()}")
print(f"✓ GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# List uploaded files
import os
files = os.listdir('/content/')
print(f"Files in /content/: {files}")

# Check for chunks.json
if 'chunks.json' in files:
    import json
    with open('chunks.json') as f:
        chunks = json.load(f)
    print(f"✓ Loaded {len(chunks)} chunks from chunks.json")
else:
    print("⚠️ chunks.json not found! Please upload it using the file icon on the left.")

## Step 3: Load AI Model (Qwen2.5-3B - Fast & Efficient)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Use smaller model for faster processing on free T4
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print(f"✓ Model loaded on GPU")

## Step 4: Define Processing Functions

In [ ]:
import json
import re

# Valid Islamic topics
VALID_TOPICS = [
    "Tawheed (Oneness of Allah)",
    "Salah (Prayer)",
    "Zakat (Charity)",
    "Sawm (Fasting)",
    "Hajj (Pilgrimage)",
    "Akhlaq (Character/Manners)",
    "Sabr (Patience)",
    "Shukr (Gratitude)",
    "Love/Mercy",
    "Forgiveness",
    "Knowledge/Wisdom",
    "Death/Afterlife",
    "Dua (Supplication)",
    "Justice/Oppression",
    "Sin/Repentance",
    "Quran/Sunnah",
    "Other"
]

def process_chunk(text):
    """Process a single chunk - generate summary and topic."""
    
    prompt = f"""You are analyzing Islamic lecture transcripts. For the following text:

1. Write a 1-2 sentence summary of the main Islamic teaching or lesson
2. Choose ONE primary topic from this list: {', '.join(VALID_TOPICS)}

Text:
{text[:800]}

Respond in this exact JSON format:
{{
  "summary": "your summary here",
  "topic": "exact topic name from list",
  "confidence": 0.95
}}

JSON:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract JSON from response
    json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
    if json_match:
        try:
            result = json.loads(json_match.group())
            # Validate topic
            if result.get('topic') not in VALID_TOPICS:
                result['topic'] = 'Other'
            return result
        except:
            pass
    
    # Fallback
    return {
        "summary": response.strip()[:200],
        "topic": "Other",
        "confidence": 0.5
    }


def process_batch(chunks, batch_size=8):
    """Process chunks in batches for GPU efficiency."""
    results = []
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        batch_num = (i // batch_size) + 1
        total_batches = (len(chunks) + batch_size - 1) // batch_size
        
        print(f"Processing batch {batch_num}/{total_batches}...")
        
        for chunk in batch:
            try:
                result = process_chunk(chunk['text'])
                chunk['summary'] = result['summary']
                chunk['topic'] = result['topic']
                chunk['confidence'] = result.get('confidence', 0.8)
                results.append(chunk)
                print(f"  ✓ Chunk {len(results)}/{len(chunks)} - Topic: {result['topic']}")
            except Exception as e:
                print(f"  ✗ Error: {e}")
                chunk['summary'] = "Processing error"
                chunk['topic'] = "Other"
                chunk['confidence'] = 0.0
                results.append(chunk)
        
        # Save checkpoint every 5 batches
        if batch_num % 5 == 0:
            with open('processed_chunks_checkpoint.json', 'w') as f:
                json.dump(results, f, indent=2)
            print(f"  💾 Checkpoint saved ({len(results)} chunks)")
    
    return results

## Step 5: Process All Chunks

In [ ]:
import json
import time

# Load chunks
with open('chunks.json') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")
print(f"Starting GPU processing...")
print(f"Estimated time: ~{len(chunks) * 2} seconds ({len(chunks) * 2 / 60:.1f} minutes)")
print()

# Process
start_time = time.time()
results = process_batch(chunks, batch_size=8)
elapsed = time.time() - start_time

print()
print(f"✓ Processing complete in {elapsed/60:.1f} minutes")
print(f"✓ Processed {len(results)} chunks")

## Step 6: Save & Download Results

In [ ]:
import json

# Save final results
with open('processed_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Saved to: processed_chunks.json")

# Show statistics
from collections import Counter
topics = Counter([r['topic'] for r in results])
print(f"\nTopic distribution:")
for topic, count in topics.most_common():
    print(f"  {topic}: {count}")

In [ ]:
# Download results
from google.colab import files

print("Downloading processed_chunks.json...")
files.download('processed_chunks.json')

print()
print("="*60)
print("NEXT STEP:")
print("="*60)
print("1. The file will download automatically")
print("2. On your computer, run:")
print("   python import_results.py processed_chunks.json")
print("="*60)